In [ ]:
import pandas as pd
import os

In [ ]:
# load the downloaded indices file
indices_path = '../data/raw/covid_id_indices.csv'
df_indices = pd.read_csv(indices_path)

# Define the available EU capitals based on how they are spelled in the dataset
eu_capitals_search = [
    'Wien', 'Bruxelles', 'Praha', 'København', 'Paris', 'Berlin', 
    'Roma', 'Madrid', 'Stockholm', 'Dublin', 'Lisboa', 
    'Grad Zagreb', 'Riga', 'Amsterdam', 'Vilniaus m.'
]

# Filter indices to extract these specific cities
capitals_mask = (
    df_indices['administrative_area_level_2'].isin(eu_capitals_search) | 
    df_indices['administrative_area_level_3'].isin(eu_capitals_search)
)
df_capitals = df_indices[capitals_mask].copy()

# Some cities (like Berlin) include dozens of sub-districts at level 3. 
# To get just the main city, we sort by level and drop the sub-district duplicates.
df_capitals = df_capitals.sort_values('administrative_area_level').drop_duplicates(subset=['administrative_area_level_1'])

# Extract the correct name (whether it was stored in level 2 or level 3) and map it to the ID
def get_city_name(row):
    if pd.notna(row['administrative_area_level_3']) and row['administrative_area_level_3'] in eu_capitals_search:
        return row['administrative_area_level_3']
    return row['administrative_area_level_2']

df_capitals['city_name'] = df_capitals.apply(get_city_name, axis=1)
capital_id_mapping = dict(zip(df_capitals['id'], df_capitals['city_name']))

print(f"Found {len(capital_id_mapping)} EU Capitals with city-level data.")

Found 16 EU Capitals with city-level data.


In [ ]:
# Define the relative path to your data
# '..' moves up from 'code' to the project root
file_path = os.path.join('..', 'data', 'raw', 'covid_data.csv')

pd.set_option('display.max_columns', None)

# Load and display the first few rows
try:
    df = pd.read_csv(file_path)
    print("Data loaded successfully!")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file was not found at {file_path}")
    print(f"Current working directory: {os.getcwd()}")

C:\Users\csehi\AppData\Local\Temp\ipykernel_49752\849855250.py:10: DtypeWarning: Columns (41,42,43,45) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Data loaded successfully!


,id,date,confirmed,deaths,recovered,tests,vaccines,people_vaccinated,people_fully_vaccinated,hosp,...,iso_alpha_3,iso_alpha_2,iso_numeric,iso_currency,key_local,key_google_mobility,key_apple_mobility,key_jhu_csse,key_nuts,key_gadm
0,0007cb93,2020-03-30,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,USA,US,840,USD,13249,ChIJzXKpmryy84gRdueJavHVBtk,NaN,US13249,NaN,USA.11.123_1
1,0007cb93,2020-03-31,4.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,USA,US,840,USD,13249,ChIJzXKpmryy84gRdueJavHVBtk,NaN,US13249,NaN,USA.11.123_1
2,0007cb93,2020-04-01,4.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,USA,US,840,USD,13249,ChIJzXKpmryy84gRdueJavHVBtk,NaN,US13249,NaN,USA.11.123_1
3,0007cb93,2020-04-02,4.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,USA,US,840,USD,13249,ChIJzXKpmryy84gRdueJavHVBtk,NaN,US13249,NaN,USA.11.123_1
4,0007cb93,2020-04-03,5.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,USA,US,840,USD,13249,ChIJzXKpmryy84gRdueJavHVBtk,NaN,US13249,NaN,USA.11.123_1


In [4]:
target_columns = [
    'id', 
    'date', 
    'school_closing', 
    'workplace_closing', 
    'cancel_events', 
    'gatherings_restrictions', 
    'transport_closing', 
    'stay_home_restrictions', 
    'internal_movement_restrictions', 
    'international_movement_restrictions'
]

In [5]:
# Filter the dataframe to only keep the 15 EU Capitals
df_filtered = df[df['id'].isin(capital_id_mapping.keys())].copy()

# Map the ID to the city name and reorganize columns
df_filtered['city_name'] = df_filtered['id'].map(capital_id_mapping)

# Move 'city_name' to the front and drop the raw 'id'
final_cols = ['city_name', 'date'] + target_columns[2:]
df_final = df_filtered[final_cols]

In [ ]:
output_path = '../data/clean/eu_capitals_covid_data.csv'
df_final.to_csv(output_path, index=False)
print(f"Success! Filtered data saved to: {output_path}")

Success! Filtered data saved to: ../data/clean/eu_capitals_policy_data.csv


In [9]:
# Preview the final dataframe
pd.set_option('display.max_columns', None)
display(df_final.head())

,city_name,date,school_closing,workplace_closing,cancel_events,gatherings_restrictions,transport_closing,stay_home_restrictions,internal_movement_restrictions,international_movement_restrictions
792968,Amsterdam,2020-02-28,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
792969,Amsterdam,2020-02-29,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
792970,Amsterdam,2020-03-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
792971,Amsterdam,2020-03-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
792972,Amsterdam,2020-03-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
